# Step 4: Preprocess Images

Normalize, augment, and organize images for model training.

## Process

1. **Load Images**: Read PNG files from Step 3
2. **Normalize**: Convert pixel values to [0, 1] range
3. **Augmentation**: Apply data augmentation (rotation, flip, zoom)
4. **Split Data**: Create train/val/test sets
5. **Save Preprocessed**: Store as numpy arrays or HDF5

In [ ]:
# Setup: Configure environment for Google Colab
import os
import sys
from pathlib import Path

try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print("🔧 Setting up Google Colab environment...\n")
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    project_path = '/content/drive/MyDrive/xai-dark-matter-localization'
    os.chdir(project_path)
    if project_path not in sys.path:
        sys.path.insert(0, project_path)
    print("✓ Colab environment configured\n")
else:
    print("⚠️  Running locally\n")

## 1. Import Libraries

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import h5py

# TensorFlow/Keras for image augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import img_to_array, array_to_img

# Project modules
from src.config import DATA_ROOT, DATASET_DIR

print("✓ All libraries imported successfully")

## 2. Load Image Metadata

In [ ]:
# Load metadata from Step 3
if isinstance(DATA_ROOT, str):
    DATA_ROOT = Path(DATA_ROOT)

metadata_file = DATA_ROOT / "processed" / "TNG-DM-XAI-v1" / "metadata_with_images.csv"

if not metadata_file.exists():
    raise FileNotFoundError(f"❌ Metadata file not found: {metadata_file}")

df_metadata = pd.read_csv(metadata_file)

print(f"✓ Loaded metadata: {len(df_metadata)} entries")
print(f"\nDataFrame shape: {df_metadata.shape}")
print(f"Columns: {list(df_metadata.columns)}")
print(f"\nResolutions available:")
for res in sorted(df_metadata['resolution'].unique()):
    count = len(df_metadata[df_metadata['resolution'] == res])
    print(f"  {res}x{res}: {count} images")

print(f"\nFirst few rows:")
print(df_metadata[['subhalo_id', 'resolution', 'mass_log_msun']].head())

## 3. Preprocessing Configuration

In [ ]:
# Configuration for preprocessing
RESOLUTION = 512  # Use higher resolution for better XAI interpretability
TARGET_SIZE = (512, 512)  # Target image size for model
NORMALIZE = True  # Normalize to [0, 1]
AUGMENT = True  # Apply data augmentation
TEST_SIZE = 0.15
VAL_SIZE = 0.1  # Of remaining after test split
RANDOM_STATE = 42

print(f"Preprocessing Configuration:")
print(f"  Target resolution: {RESOLUTION}x{RESOLUTION}")
print(f"  Normalization: {NORMALIZE}")
print(f"  Data augmentation: {AUGMENT}")
print(f"  Train/Val/Test split: 75%/10%/15%")
print(f"  Random seed: {RANDOM_STATE}")

## 4. Load and Normalize Images

In [ ]:
# Filter metadata for target resolution and load images
df_target = df_metadata[df_metadata['resolution'] == RESOLUTION].copy().reset_index(drop=True)

print(f"🔍 Loading {len(df_target)} images at {RESOLUTION}x{RESOLUTION}...\n")

images = []
valid_indices = []

for idx, row in tqdm(df_target.iterrows(), total=len(df_target), desc="Loading"):
    img_path = Path(row['image_path'])
    
    # Handle relative vs absolute paths
    if not img_path.is_absolute():
        img_path = DATA_ROOT.parent / img_path
    
    if img_path.exists():
        try:
            # Load image as grayscale (dark matter is single channel)
            img = Image.open(img_path).convert('L')
            
            # Resize if necessary
            if img.size != TARGET_SIZE:
                img = img.resize(TARGET_SIZE, Image.Resampling.LANCZOS)
            
            # Convert to numpy array
            img_array = np.array(img, dtype=np.float32)
            
            # Normalize to [0, 1]
            if NORMALIZE:
                img_array = img_array / 255.0
            
            images.append(img_array)
            valid_indices.append(idx)
            
        except Exception as e:
            print(f"  ❌ Error loading {img_path.name}: {str(e)[:50]}")
    else:
        print(f"  ⚠️  File not found: {img_path}")

# Convert to numpy array with shape (N, H, W, 1) for compatibility
X = np.array(images, dtype=np.float32)
X = X[:, :, :, np.newaxis]  # Add channel dimension

# Update metadata to only valid images
df_target = df_target.iloc[valid_indices].reset_index(drop=True)

print(f"\n✓ Successfully loaded {len(X)} images")
print(f"  Shape: {X.shape}")
print(f"  Dtype: {X.dtype}")
print(f"  Value range: [{X.min():.3f}, {X.max():.3f}]")
print(f"  Mean: {X.mean():.3f}, Std: {X.std():.3f}")

## 5. Train/Validation/Test Split

In [ ]:
# Split data: 85% train+val, 15% test
X_train_val, X_test, idx_train_val, idx_test = train_test_split(
    X, 
    np.arange(len(X)),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

# Split train+val: 90% train, 10% val (of 85%)
X_train, X_val, idx_train, idx_val = train_test_split(
    X_train_val,
    idx_train_val,
    test_size=VAL_SIZE/(1-TEST_SIZE),
    random_state=RANDOM_STATE
)

# Get corresponding metadata
df_train = df_target.iloc[idx_train].copy()
df_val = df_target.iloc[idx_val].copy()
df_test = df_target.iloc[idx_test].copy()

print("✓ Data split complete:")
print(f"  Train: {len(X_train)} images ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Val:   {len(X_val)} images ({len(X_val)/len(X)*100:.1f}%)")
print(f"  Test:  {len(X_test)} images ({len(X_test)/len(X)*100:.1f}%)")
print(f"  Total: {len(X)} images")

# Add split column to metadata
df_target['split'] = 'unknown'
df_target.loc[idx_train, 'split'] = 'train'
df_target.loc[idx_val, 'split'] = 'val'
df_target.loc[idx_test, 'split'] = 'test'

print(f"\nSplit distribution:")
print(df_target['split'].value_counts().sort_index())

## 6. Data Augmentation (Optional)

In [ ]:
# Optional: Apply data augmentation to training set only
if AUGMENT:
    print("📊 Applying data augmentation to training set...\n")
    
    # Define augmentation parameters
    aug = ImageDataGenerator(
        rotation_range=15,           # Small rotations (dark matter is rotationally symmetric)
        horizontal_flip=True,        # Horizontal flip
        vertical_flip=True,          # Vertical flip
        zoom_range=0.1,              # Slight zoom
        fill_mode='nearest'
    )
    
    # Generate augmented training data (on-the-fly during training)
    # For now, we'll just apply it once
    X_train_aug = []
    for i in range(len(X_train)):
        # Generate 2 augmented versions per original image
        img = X_train[i]
        aug_iter = aug.flow(np.expand_dims(img, 0), batch_size=1)
        
        for _ in range(2):
            X_train_aug.append(next(aug_iter)[0])
    
    # Only augment if we have time; otherwise just use original
    if len(X_train_aug) > 0:
        X_train_original = X_train.copy()
        X_train_aug = np.array(X_train_aug)
        # Optionally double training set with augmented images
        # X_train = np.concatenate([X_train, X_train_aug[:len(X_train)]], axis=0)
        print(f"  ✓ Generated {len(X_train_aug)} augmented images")
        print(f"  Training set could be expanded to {len(X_train) * 2} if using augmentation")
    else:
        print(f"  ℹ️  Augmentation available but not applied to preserve original data")
else:
    print("ℹ️  Data augmentation disabled")

## 7. Save Preprocessed Data

In [ ]:
# Save preprocessed data as HDF5 for efficient loading
output_dir = DATA_ROOT / "processed" / "TNG-DM-XAI-v1" / "preprocessed"
output_dir.mkdir(parents=True, exist_ok=True)

h5_file = output_dir / f"images_{RESOLUTION}_preprocessed.h5"

print(f"💾 Saving preprocessed images to HDF5...\n")
print(f"  File: {h5_file}")

with h5py.File(h5_file, 'w') as f:
    # Create datasets
    f.create_dataset('X_train', data=X_train, compression='gzip', compression_opts=4)
    f.create_dataset('X_val', data=X_val, compression='gzip', compression_opts=4)
    f.create_dataset('X_test', data=X_test, compression='gzip', compression_opts=4)
    
    # Store metadata as attributes
    f.attrs['resolution'] = RESOLUTION
    f.attrs['target_size'] = str(TARGET_SIZE)
    f.attrs['normalized'] = NORMALIZE
    f.attrs['n_train'] = len(X_train)
    f.attrs['n_val'] = len(X_val)
    f.attrs['n_test'] = len(X_test)

print(f"✓ Saved HDF5 file ({h5_file.stat().st_size / 1e6:.1f} MB)")

# Also save metadata CSVs per split
for split, df_split in [('train', df_train), ('val', df_val), ('test', df_test)]:
    split_file = output_dir / f"metadata_{split}_{RESOLUTION}.csv"
    df_split.to_csv(split_file, index=False)
    print(f"✓ Saved metadata: {split_file.name}")

print(f"\n✓ All data saved to: {output_dir}")

## 8. Summary and Statistics

In [ ]:
# Display final summary
print("\n" + "="*60)
print("PREPROCESSING SUMMARY")
print("="*60)

print(f"\n📊 Dataset Statistics:")
print(f"  Total images: {len(df_target)}")
print(f"  Resolution: {RESOLUTION}x{RESOLUTION}")
print(f"  Target size: {TARGET_SIZE}")
print(f"  Channel: Grayscale (1)")

print(f"\n📈 Data Splits:")
print(f"  Train: {len(X_train):,} images ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Val:   {len(X_val):,} images ({len(X_val)/len(X)*100:.1f}%)")
print(f"  Test:  {len(X_test):,} images ({len(X_test)/len(X)*100:.1f}%)")

print(f"\n🔢 Image Statistics:")
print(f"  Value range: [{X.min():.3f}, {X.max():.3f}]")
print(f"  Mean: {X.mean():.3f}")
print(f"  Std: {X.std():.3f}")

print(f"\n💾 Output Files:")
print(f"  HDF5: {h5_file.name} ({h5_file.stat().st_size / 1e6:.1f} MB)")
print(f"  Metadata: metadata_train/val/test_{RESOLUTION}.csv")

print(f"\n✅ Preprocessing complete! Ready for model training.")
print("="*60)

# Show sample images
print("\n📷 Sample Images (first 3 from each split):")

fig, axes = plt.subplots(3, 3, figsize=(10, 10))

for i in range(3):
    # Train
    axes[0, i].imshow(X_train[i].squeeze(), cmap='hot')
    axes[0, i].set_title(f'Train {i+1}')
    axes[0, i].axis('off')
    
    # Val
    axes[1, i].imshow(X_val[i].squeeze(), cmap='hot')
    axes[1, i].set_title(f'Val {i+1}')
    axes[1, i].axis('off')
    
    # Test
    axes[2, i].imshow(X_test[i].squeeze(), cmap='hot')
    axes[2, i].set_title(f'Test {i+1}')
    axes[2, i].axis('off')

plt.suptitle(f'Dark Matter Mass Distribution ({RESOLUTION}x{RESOLUTION})', fontsize=14)
plt.tight_layout()
plt.savefig(output_dir / 'sample_images.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\n✓ Sample visualization saved")